In [7]:
import os
import warnings

from datetime import datetime, timedelta
from importlib import reload

import hopsworks
import pandas as pd

from dotenv import load_dotenv
from hopsworks.project import Project
from hsfs.feature_group import FeatureGroup
from hsfs.feature_store import FeatureStore

import src.ingest
reload(src.ingest)

from src.ingest import fetch_synthetic_batch

warnings.filterwarnings("ignore")
load_dotenv()

True

In [8]:
# get the starting and ending timestamps for the synthetic batch of data
# NOTE: the synthetic batch will included data from the previous 7 days up until now
end: pd.Timestamp = pd.Timestamp(datetime.now()).floor("H")
start: pd.Timestamp = end - timedelta(days=7)
df_batch: pd.DataFrame = fetch_synthetic_batch(start, end)
df_batch

  0%|          | 0/12 [00:00<?, ?it/s]

2024-05-12 16:52:44,287 INFO: rides_2022-01.parquet exists. Skipping download.
2024-05-12 16:52:44,288 INFO: rides_2022-02.parquet exists. Skipping download.
2024-05-12 16:52:44,289 INFO: rides_2022-03.parquet exists. Skipping download.
2024-05-12 16:52:44,290 INFO: rides_2022-04.parquet exists. Skipping download.
2024-05-12 16:52:44,290 INFO: rides_2022-05.parquet exists. Skipping download.
2024-05-12 16:52:44,291 INFO: rides_2022-06.parquet exists. Skipping download.
2024-05-12 16:52:44,291 INFO: rides_2022-07.parquet exists. Skipping download.
2024-05-12 16:52:44,292 INFO: rides_2022-08.parquet exists. Skipping download.
2024-05-12 16:52:44,292 INFO: rides_2022-09.parquet exists. Skipping download.
2024-05-12 16:52:44,293 INFO: rides_2022-10.parquet exists. Skipping download.
2024-05-12 16:52:44,293 INFO: rides_2022-11.parquet exists. Skipping download.
2024-05-12 16:52:44,294 INFO: rides_2022-12.parquet exists. Skipping download.


100%|██████████| 12/12 [00:00<00:00, 1681.59it/s]


,pickup_location_id,pickup_hour,n_taxi_rides
0,1,2024-05-05 16:00:00,1.0
1,1,2024-05-05 17:00:00,1.0
2,1,2024-05-05 18:00:00,0.0
3,1,2024-05-05 19:00:00,5.0
4,1,2024-05-05 20:00:00,2.0
...,...,...,...
42515,265,2024-05-12 12:00:00,5.0
42516,265,2024-05-12 13:00:00,4.0
42517,265,2024-05-12 14:00:00,5.0
42518,265,2024-05-12 15:00:00,10.0


In [ ]:
# # create a new column called 'pickup_ts', that converts each timestamp in the 'pickup_hour' ...
# # column to a value that represents unix epoch milliseconds
# # NOTE: the reference site is, https://currentmillis.com/
# cols: list[str] = df_batch.columns.tolist()
# cols.insert(2, "pickup_ts")
# df_batch = (
#     df_batch
#     .assign(pickup_ts=df_batch["pickup_hour"].astype(int) // 1_000_000)
#     [cols]
# )
# df_batch

In [11]:
# login to Hopsworks and connect to the 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project=os.getenv("HOPSWORKS_PROJECT_NAME"), 
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# connect to the project's feature store
feature_store: FeatureStore = project.get_feature_store()

Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.


In [12]:
# create the feature group, which will be used to write data to the feature store
# NOTE: the 'primary_key' parameter is used as a unique row identifier, that is, ...
# each row will be identified based on its pickup location ID and pickup hour
feature_group: FeatureGroup = feature_store.get_or_create_feature_group(
    name="preprocessed_data", 
    version=1, 
    description="NYC taxi rides at hourly frequency", 
    primary_key=["pickup_location_id", "pickup_hour"], 
    event_time="pickup_hour"
)

# write the 'df_batch' pd.DataFrame to the 'preprocessed_data' feature group
feature_group.insert(df_batch, write_options={"wait_for_job": False})

Feature Group created successfully, explore it at 
https://c.app.hopsworks.ai:443/p/708756/fs/704579/fg/808267


Uploading Dataframe: 0.00% |          | Rows 0/42520 | Elapsed Time: 00:00 | Remaining Time: ?

Launching job: preprocessed_data_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai/p/708756/jobs/named/preprocessed_data_1_offline_fg_materialization/executions


(<hsfs.core.job.Job at 0x138829810>, None)